# Notebook 03: Silver Cleaning
## Purpose: Clean raw sensor data, engineer RUL targets, write silver.sensor_cleaned
## Input:  workspace.predictive_maintenance.bronze_nasa_sensor_raw
## Output: workspace.predictive_maintenance.silver_sensor_cleaned
## Key Steps: Drop low-variance sensors, handle nulls, cast types, deduplicate, engineer targets


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.sql.window import Window

# Load bronze table
df = spark.table("workspace.predictive_maintenance.bronze_nasa_sensor_raw")

print(f"Loaded: {df.count():,} rows | {len(df.columns)} columns")
display(df.limit(3))

In [0]:
# Keep only top 7 high-variance sensors identified in EDA
# Dropping 14 near-zero variance sensors

SENSORS_TO_KEEP = [
    'sensor_3', 'sensor_4', 'sensor_7', 
    'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12'
]

SENSORS_TO_DROP = [f'sensor_{i}' for i in range(1, 22) 
                   if f'sensor_{i}' not in SENSORS_TO_KEEP]

# Drop low variance sensors
df_clean = df.drop(*SENSORS_TO_DROP)

print(f"Sensors kept:    {len(SENSORS_TO_KEEP)}")
print(f"Sensors dropped: {len(SENSORS_TO_DROP)}")
print(f"Columns now:     {len(df_clean.columns)}")
print(f"Kept: {SENSORS_TO_KEEP}")
print(f"Dropped: {SENSORS_TO_DROP}")

In [0]:
# Cast sensor columns to DoubleType
for s in SENSORS_TO_KEEP:
    df_clean = df_clean.withColumn(s, F.col(s).cast(DoubleType()))

# Cast unit_id and cycle to IntegerType
df_clean = df_clean \
    .withColumn("unit_id", F.col("unit_id").cast(IntegerType())) \
    .withColumn("cycle",   F.col("cycle").cast(IntegerType()))

# Cast settings to DoubleType
for s in ['setting_1', 'setting_2', 'setting_3']:
    df_clean = df_clean.withColumn(s, F.col(s).cast(DoubleType()))

print("Schema after casting:")
df_clean.printSchema()

In [0]:
# Check nulls before
print("=== NULL COUNTS BEFORE ===")
null_counts = df_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_clean.columns
])
null_counts.show()

# Forward fill nulls per unit_id using Window
window_spec = Window.partitionBy("unit_id").orderBy("cycle") \
                    .rowsBetween(Window.unboundedPreceding, 0)

for s in SENSORS_TO_KEEP:
    df_clean = df_clean.withColumn(
        s, F.last(F.col(s), ignorenulls=True).over(window_spec)
    )

# Drop any remaining nulls
rows_before = df_clean.count()
df_clean = df_clean.dropna()
rows_after = df_clean.count()

print(f"\nRows before dropna: {rows_before:,}")
print(f"Rows after dropna:  {rows_after:,}")
print(f"Rows removed:       {rows_before - rows_after:,}")

In [0]:
rows_before = df_clean.count()

# Remove duplicate unit_id + cycle combinations
df_clean = df_clean.dropDuplicates(["unit_id", "cycle"])

rows_after = df_clean.count()

print(f"Rows before dedup: {rows_before:,}")
print(f"Rows after dedup:  {rows_after:,}")
print(f"Duplicates removed: {rows_before - rows_after:,}")

In [0]:
# Calculate max cycle per machine
window_max = Window.partitionBy("unit_id")

df_clean = df_clean.withColumn(
    "max_cycle", F.max("cycle").over(window_max)
)

# RUL = max_cycle - current_cycle
df_clean = df_clean.withColumn(
    "RUL", F.col("max_cycle") - F.col("cycle")
)

# Verify: RUL at last cycle should be 0
print("=== RUL VERIFICATION ===")
df_clean.filter(
    F.col("cycle") == F.col("max_cycle")
).select("unit_id", "cycle", "max_cycle", "RUL") \
 .show(5)

print(f"Min RUL: {df_clean.agg(F.min('RUL')).collect()[0][0]}")
print(f"Max RUL: {df_clean.agg(F.max('RUL')).collect()[0][0]}")
print(f"RUL = 0 at last cycle confirmed ")

In [0]:
# Binary classification targets
df_clean = df_clean \
    .withColumn("fail_30", (F.col("RUL") <= 30).cast(IntegerType())) \
    .withColumn("fail_15", (F.col("RUL") <= 15).cast(IntegerType()))

# Check class balance
total = df_clean.count()
fail30 = df_clean.filter(F.col("fail_30") == 1).count()
fail15 = df_clean.filter(F.col("fail_15") == 1).count()

print("=== CLASS BALANCE ===")
print(f"Total rows:          {total:,}")
print(f"fail_30 positive:    {fail30:,} ({fail30/total*100:.1f}%)")
print(f"fail_15 positive:    {fail15:,} ({fail15/total*100:.1f}%)")
print("\nUsing F1-score as primary metric — not accuracy")
print("Reason: Class imbalance means accuracy is misleading")

In [0]:
# Drop max_cycle helper column before writing
df_silver = df_clean.drop("max_cycle")

# Write as Delta table
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.predictive_maintenance.silver_sensor_cleaned")

# Verify
count = spark.table("workspace.predictive_maintenance.silver_sensor_cleaned").count()
cols  = len(spark.table("workspace.predictive_maintenance.silver_sensor_cleaned").columns)

print(f"   silver_sensor_cleaned written!")
print(f"   Rows:    {count:,}")
print(f"   Columns: {cols}")